# Set-Piece Dependency & Efficiency: EPL 2025/26

Analisis distribusi Expected Goals (xG) tim-tim Premier League musim 2025/26 berdasarkan skema permainan, untuk mengukur **seberapa bergantung** tim pada bola mati dan **seberapa efisien** mereka mengonversi peluang itu jadi gol. Termasuk pembanding singkat dengan La Liga.

**Sumber data:** [Understat](https://understat.com) via library `understatapi`.

In [ ]:
!pip install understatapi

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from understatapi import UnderstatClient

## 1. Eksplorasi struktur data

Sebelum scraping semua tim, kita coba 1 tim dulu (`Arsenal`) untuk memahami bentuk data yang dikembalikan `get_context_data()` — supaya kode berikutnya dibangun di atas struktur data yang benar-benar terverifikasi, bukan asumsi.

In [ ]:
with UnderstatClient() as understat:
    context_data = understat.team(team="Arsenal").get_context_data(season="2025")

context_data.keys()

In [ ]:
context_data["situation"]

`situation` berbentuk dictionary: key-nya nama kategori situasi (`OpenPlay`, `FromCorner`, `SetPiece`, `DirectFreekick`, `Penalty`), value-nya berisi `shots`, `goals`, `xG` untuk kategori itu.

## 2. Ambil daftar tim EPL musim 2025/26

Daftar tim diambil secara **dinamis** lewat `LeagueEndpoint.get_team_data()`, bukan hardcode manual — karena komposisi 20 tim EPL berubah tiap musim (promosi/degradasi). Cara ini otomatis menyesuaikan musim yang diminta.

In [ ]:
with UnderstatClient() as understat:
    league_teams = understat.league(league="EPL").get_team_data(season="2025")

team_names = [info["title"] for info in league_teams.values()]
team_names

## 3. Scraping data situasi untuk semua tim

Loop tiap tim, dengan tiga pertimbangan penting:
- **`team.replace(" ", "_")`** — menyesuaikan format slug URL Understat (pakai underscore)
- **`try/except`** — kalau 1 tim gagal di-scrape, loop tetap lanjut ke tim lain, bukan berhenti total
- **`time.sleep(1)`** — jeda antar request supaya tidak membebani/memicu rate-limit server Understat

In [ ]:
all_situations = []

with UnderstatClient() as understat:
    for team in team_names:
        team_slug = team.replace(" ", "_")
        try:
            ctx = understat.team(team=team_slug).get_context_data(season="2025")
            df_team = pd.DataFrame.from_dict(ctx["situation"], orient="index")
            df_team.index.name = "situation"
            df_team = df_team.reset_index()
            df_team["team"] = team
            all_situations.append(df_team)
        except Exception as e:
            print(f"Gagal ambil {team}: {e}")
        time.sleep(1)

df_all = pd.concat(all_situations, ignore_index=True)
df_all.head()

## 4. Hitung Set-Piece Dependency %

Metrik utama: `SetPiece_Dependency_%` = proporsi xG dari situasi bola mati (Corner, Set Piece, Direct Free-kick, Penalty) dibanding total xG tim. Dipakai **persentase**, bukan angka xG mentah, supaya perbandingan adil antar tim besar dan kecil — tim yang volumenya besar tidak otomatis terlihat "lebih bergantung" hanya karena angka mentahnya besar.

In [ ]:
total_xg = df_all.groupby("team")["xG"].sum().rename("total_xG")

dead_ball_situations = ["FromCorner", "SetPiece", "DirectFreekick", "Penalty"]
setpiece_xg = (
    df_all[df_all["situation"].isin(dead_ball_situations)]
    .groupby("team")["xG"].sum()
    .rename("setpiece_xG")
)

dependency = pd.concat([total_xg, setpiece_xg], axis=1).fillna(0)
dependency["SetPiece_Dependency_%"] = (dependency["setpiece_xG"] / dependency["total_xG"] * 100).round(2)
dependency.sort_values("SetPiece_Dependency_%", ascending=False)

In [ ]:
dependency.to_csv("epl_setpiece_dependency_2025_26.csv")

## 5. Bandingkan xG dengan gol aktual (efisiensi konversi)

xG tinggi tidak selalu berbuah gol. Di sini kita hitung porsi **gol nyata** dari bola mati (`SetPiece_Goal_%`), lalu bandingkan dengan `SetPiece_Dependency_%` (berbasis xG) lewat `xG_vs_Goal_gap`. Gap positif = tim overperform/klinis di bola mati, gap negatif = tim underperform/boros peluang.

In [ ]:
goals_pivot = df_all.groupby(["team", "situation"])["goals"].sum().unstack(fill_value=0)

dead_ball_situations = ["FromCorner", "SetPiece", "DirectFreekick", "Penalty"]
goals_pivot["OpenPlay_goals"] = goals_pivot["OpenPlay"]
goals_pivot["SetPiece_goals"] = goals_pivot[dead_ball_situations].sum(axis=1)

goals_comparison = goals_pivot[["OpenPlay_goals", "SetPiece_goals"]].copy()
goals_comparison["total_goals"] = goals_comparison["OpenPlay_goals"] + goals_comparison["SetPiece_goals"]
goals_comparison["SetPiece_Goal_%"] = (goals_comparison["SetPiece_goals"] / goals_comparison["total_goals"] * 100).round(2)

goals_comparison.sort_values("SetPiece_Goal_%", ascending=False)

In [ ]:
comparison = dependency[["SetPiece_Dependency_%"]].join(goals_comparison[["SetPiece_Goal_%", "OpenPlay_goals", "SetPiece_goals"]])
comparison["xG_vs_Goal_gap"] = (comparison["SetPiece_Goal_%"] - comparison["SetPiece_Dependency_%"]).round(2)
comparison.sort_values("xG_vs_Goal_gap", ascending=False)

## 6. Visualisasi 1 — Ranking Dependency % (chart pembuka)

Horizontal bar chart sederhana, warna kondisional (merah = di atas rata-rata liga, biru = di bawah) supaya bisa dipahami dalam hitungan detik — berfungsi sebagai pembuka sebelum masuk ke analisis efisiensi yang lebih dalam.

In [ ]:
plot_data = dependency.sort_values("SetPiece_Dependency_%", ascending=True)

fig, ax = plt.subplots(figsize=(9, 10))

colors = ["#e34a33" if v >= plot_data["SetPiece_Dependency_%"].mean() else "#2b8cbe"
          for v in plot_data["SetPiece_Dependency_%"]]

ax.barh(plot_data.index, plot_data["SetPiece_Dependency_%"], color=colors)

mean_val = plot_data["SetPiece_Dependency_%"].mean()
ax.axvline(mean_val, color="gray", linestyle="--", linewidth=1)
ax.text(mean_val + 0.5, -0.8, f"Rata-rata liga: {mean_val:.1f}%", fontsize=9, color="gray")

ax.set_xlabel("Set-Piece Dependency %")
ax.set_title("Set-Piece Dependency Ranking — EPL 2025/26")

plt.tight_layout()
plt.savefig("setpiece_dependency_ranking.png", dpi=200)
plt.show()

## 7. Pembanding La Liga

Logika scraping + hitung dependency % dibungkus jadi **function** yang menerima parameter `league_name` dan `season` — supaya bisa dipakai ulang untuk liga lain tanpa duplikasi kode (prinsip DRY: *Don't Repeat Yourself*).

In [ ]:
def get_league_setpiece_dependency(league_name, season, dead_ball_situations=None):
    """
    Scrape data situational xG untuk semua tim di sebuah liga & musim,
    lalu hitung Set-Piece Dependency % per tim.
    """
    if dead_ball_situations is None:
        dead_ball_situations = ["FromCorner", "SetPiece", "DirectFreekick", "Penalty"]

    with UnderstatClient() as understat:
        league_teams = understat.league(league=league_name).get_team_data(season=season)
        team_names = [info["title"] for info in league_teams.values()]

        all_situations = []
        for team in team_names:
            team_slug = team.replace(" ", "_")
            try:
                ctx = understat.team(team=team_slug).get_context_data(season=season)
                df_team = pd.DataFrame.from_dict(ctx["situation"], orient="index")
                df_team.index.name = "situation"
                df_team = df_team.reset_index()
                df_team["team"] = team
                all_situations.append(df_team)
            except Exception as e:
                print(f"Gagal ambil {team} ({league_name}): {e}")
            time.sleep(1)

    df_league = pd.concat(all_situations, ignore_index=True)

    total_xg = df_league.groupby("team")["xG"].sum().rename("total_xG")
    setpiece_xg = (
        df_league[df_league["situation"].isin(dead_ball_situations)]
        .groupby("team")["xG"].sum()
        .rename("setpiece_xG")
    )
    dep = pd.concat([total_xg, setpiece_xg], axis=1).fillna(0)
    dep["SetPiece_Dependency_%"] = (dep["setpiece_xG"] / dep["total_xG"] * 100).round(2)
    dep["league"] = league_name

    return dep

In [ ]:
laliga_dependency = get_league_setpiece_dependency("La_Liga", "2025")

epl_avg = dependency["SetPiece_Dependency_%"].mean()
laliga_avg = laliga_dependency["SetPiece_Dependency_%"].mean()

print(f"Rata-rata Set-Piece Dependency % EPL 2025/26: {epl_avg:.2f}%")
print(f"Rata-rata Set-Piece Dependency % La Liga 2025/26: {laliga_avg:.2f}%")

## 8. Visualisasi 2 — EPL vs La Liga

Perbandingan rata-rata liga dalam satu bar chart, warna disesuaikan identitas visual masing-masing liga.

In [ ]:
leagues = ['EPL', 'La Liga']
averages = [epl_avg, laliga_avg]

sns.set_theme(style="whitegrid")
plt.figure(figsize=(8, 6))

colors = ['#38003c', '#ee1c25']

bars = plt.bar(leagues, averages, color=colors, width=0.5, edgecolor='black', linewidth=1.2)

for bar in bars:
    yval = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        yval + 0.8,
        f"{yval:.2f}%",
        ha='center',
        va='bottom',
        fontsize=12,
        fontweight='bold'
    )

plt.title('Comparison of Set-Piece Dependency % (EPL vs La Liga)', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Average Set-Piece Dependency (%)', fontsize=12)
plt.xlabel('League', fontsize=12)

plt.ylim(0, max(averages) + 10)

sns.despine()

plt.tight_layout()
plt.savefig("laliga_vs_epl_comparison.png", dpi=200)
plt.show()

## 9. Visualisasi 3 — Dumbbell Chart: xG vs Gol Aktual (analisis utama)

Chart paling penting di proyek ini. Dua titik per tim (xG bola mati vs gol aktual bola mati) dihubungkan garis, diurutkan berdasarkan Dependency %. Dipilih dibanding dual-axis chart atau scatter plot gap karena lebih intuitif dibaca — titik gol di kanan titik xG berarti klinis, di kiri berarti boros peluang — tanpa perlu memahami metrik turunan ("gap") terlebih dulu.

In [ ]:
combined = dependency[["setpiece_xG", "SetPiece_Dependency_%"]].join(
    goals_comparison[["SetPiece_goals"]]
)
combined = combined.sort_values("SetPiece_Dependency_%", ascending=False)
combined

In [ ]:
plot_data = combined.sort_values("SetPiece_Dependency_%", ascending=False)

fig, ax = plt.subplots(figsize=(10, 10))

y_pos = range(len(plot_data))

for i, team in enumerate(plot_data.index):
    ax.plot(
        [plot_data.loc[team, "setpiece_xG"], plot_data.loc[team, "SetPiece_goals"]],
        [i, i],
        color="gray", linewidth=1.5, zorder=1
    )

ax.scatter(plot_data["setpiece_xG"], y_pos, color="#2b8cbe", s=100, label="Set-Piece xG (ekspektasi)", zorder=2)
ax.scatter(plot_data["SetPiece_goals"], y_pos, color="#e34a33", s=100, label="Set-Piece Goals (kenyataan)", zorder=2)

labels = [f"{team}  ({plot_data.loc[team, 'SetPiece_Dependency_%']:.0f}% dependency)" for team in plot_data.index]
ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels)
ax.invert_yaxis()

ax.set_xlabel("Jumlah (xG atau Gol)")
ax.set_title("Set-Piece: xG vs Gol Aktual, Diurutkan Berdasarkan Dependency % (EPL 2025/26)")
ax.legend()
plt.tight_layout()
plt.savefig("setpiece_xg_vs_goals_dumbbell.png", dpi=200)
plt.show()

## Kesimpulan

- **Wolverhampton Wanderers** paling bergantung pada bola mati (43.02% dari total xG), **Liverpool** paling sedikit (18.78%).
- Ketergantungan tinggi tidak selalu berbanding lurus dengan efisiensi: **Manchester United** paling klinis mengonversi peluang bola mati jadi gol, sementara **Burnley** paling banyak membuang peluang berkualitas dari situasi yang sama.
- Rata-rata Set-Piece Dependency % EPL (29.88%) sedikit lebih tinggi dibanding La Liga (27.95%) musim ini.